In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# =========================================================
# 1. 파일 경로 설정
# =========================================================

CURRENT_DIR = Path.cwd().resolve()

candidate_dirs = [
    CURRENT_DIR / "project_data",
    CURRENT_DIR.parent / "project_data",
    CURRENT_DIR.parent.parent / "project_data",
]

DATA_DIR = next(
    (path for path in candidate_dirs if path.exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "project_data 폴더를 찾지 못했습니다.\n"
        f"현재 실행 위치: {CURRENT_DIR}"
    )

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = (
    PROCESSED_DIR
    / "all_age_commute_od_aggregated.csv"
)

print("입력 파일:", INPUT_FILE)
print("입력 파일 존재:", INPUT_FILE.exists())

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"입력 파일을 찾지 못했습니다: {INPUT_FILE}"
    )

입력 파일: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_od_aggregated.csv
입력 파일 존재: True


In [3]:
# =========================================================
# 2. OD 집계 결과 불러오기
# =========================================================

od = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
    dtype={
        "거주동 코드": "string",
        "거주동 이름": "string",
        "근무동 코드": "string",
        "근무동 이름": "string",
    },
)

print("OD 행 수:", f"{len(od):,}")
print("거주동 수:", f"{od['거주동 코드'].nunique():,}")
print("근무동 수:", f"{od['근무동 코드'].nunique():,}")

display(od.head())

OD 행 수: 164,860
거주동 수: 428
근무동 수: 428


,거주동 코드,거주동 이름,근무동 코드,근무동 이름,출근_이동량,평균_이동시간_분,평균_이동거리_m,평균_이동거리_km,원본_행수
0,11560540,여의동,11560540,여의동,972322.53,14.15,622.50,0.622,14956
1,11545510,가산동,11545510,가산동,747881.32,15.10,770.26,0.770,13075
2,11680640,역삼1동,11680640,역삼1동,677469.04,13.72,550.29,0.550,13645
3,11545610,독산1동,11545510,가산동,630743.77,20.73,1378.76,1.379,12246
4,11560535,영등포동,11560540,여의동,593958.64,21.09,1535.68,1.536,12009


In [4]:
# =========================================================
# 3. 기본 데이터 점검
# =========================================================

required_cols = [
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "출근_이동량",
    "평균_이동시간_분",
    "평균_이동거리_m",
]

missing_cols = [
    col for col in required_cols
    if col not in od.columns
]

if missing_cols:
    raise ValueError(
        f"필요한 컬럼이 없습니다: {missing_cols}"
    )


# 핵심 컬럼 결측 제거
od = od.dropna(
    subset=[
        "거주동 코드",
        "근무동 코드",
        "출근_이동량",
    ]
).copy()


# 출근 이동량이 0 이하인 행 제거
od = od.loc[
    od["출근_이동량"] > 0
].copy()


# 같은 거주동-근무동 코드 조합 중복 확인
duplicate_count = od.duplicated(
    subset=[
        "거주동 코드",
        "근무동 코드",
    ]
).sum()

print("중복 OD 수:", f"{duplicate_count:,}")

assert duplicate_count == 0, (
    "같은 거주동-근무동 코드 조합이 중복되어 있습니다."
)

print("점검 완료")

중복 OD 수: 0
점검 완료


In [5]:
# =========================================================
# 4. 거주동별 전체 출근량 계산
# =========================================================

od["거주동_전체_출근량"] = (
    od
    .groupby(
        "거주동 코드",
        observed=True,
    )["출근_이동량"]
    .transform("sum")
)


print(
    od[
        [
            "거주동 코드",
            "거주동 이름",
            "근무동 코드",
            "근무동 이름",
            "출근_이동량",
            "거주동_전체_출근량",
        ]
    ].head()
)

     거주동 코드 거주동 이름    근무동 코드 근무동 이름     출근_이동량  거주동_전체_출근량
0  11560540    여의동  11560540    여의동  972322.53  2151839.53
1  11545510    가산동  11545510    가산동  747881.32  2053987.39
2  11680640   역삼1동  11680640   역삼1동  677469.04  2308603.58
3  11545610   독산1동  11545510    가산동  630743.77  2718760.88
4  11560535   영등포동  11560540    여의동  593958.64  2496372.32


In [6]:
# =========================================================
# 5. 목적지별 출근 비중 계산
# =========================================================

od["목적지_출근비중"] = (
    od["출근_이동량"]
    / od["거주동_전체_출근량"]
)

In [7]:
# =========================================================
# 6. 목적지 순위 및 누적 비중 계산
# =========================================================

od = (
    od
    .sort_values(
        [
            "거주동 코드",
            "출근_이동량",
            "근무동 코드",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


# 거주동별 목적지 출근량 순위
od["목적지_순위"] = (
    od
    .groupby(
        "거주동 코드",
        observed=True,
    )
    .cumcount()
    + 1
)


# 거주동별 누적 출근 비중
od["누적_출근비중"] = (
    od
    .groupby(
        "거주동 코드",
        observed=True,
    )["목적지_출근비중"]
    .cumsum()
)

In [8]:
# =========================================================
# 7. 목적지 비중 합계 검증
# =========================================================

share_check = (
    od
    .groupby(
        [
            "거주동 코드",
            "거주동 이름",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        목적지_비중합=("목적지_출근비중", "sum"),
        최종_누적비중=("누적_출근비중", "max"),
        전체_출근량=("출근_이동량", "sum"),
        전체_목적지수=("근무동 코드", "nunique"),
    )
)


share_check["비중합_오차"] = (
    share_check["목적지_비중합"] - 1
).abs()


print(
    share_check[
        [
            "목적지_비중합",
            "최종_누적비중",
            "비중합_오차",
        ]
    ].describe()
)


invalid_share = share_check.loc[
    share_check["비중합_오차"] > 1e-8
]

print(
    "비중 합계가 1과 다른 거주동 수:",
    f"{len(invalid_share):,}"
)

display(share_check.head())

            목적지_비중합       최종_누적비중        비중합_오차
count  4.280000e+02  4.280000e+02  4.280000e+02
mean   1.000000e+00  1.000000e+00  1.971424e-17
std    4.683851e-17  4.683851e-17  4.247687e-17
min    1.000000e+00  1.000000e+00  0.000000e+00
25%    1.000000e+00  1.000000e+00  0.000000e+00
50%    1.000000e+00  1.000000e+00  0.000000e+00
75%    1.000000e+00  1.000000e+00  0.000000e+00
max    1.000000e+00  1.000000e+00  1.110223e-16
비중 합계가 1과 다른 거주동 수: 0


,거주동 코드,거주동 이름,목적지_비중합,최종_누적비중,전체_출근량,전체_목적지수,비중합_오차
0,11110515,청운효자동,1.0,1.0,564412.04,364,1.110223e-16
1,11110530,사직동,1.0,1.0,636380.75,409,0.000000e+00
2,11110540,삼청동,1.0,1.0,106609.77,286,0.000000e+00
3,11110550,부암동,1.0,1.0,452322.26,358,0.000000e+00
4,11110560,평창동,1.0,1.0,703357.45,379,0.000000e+00


In [9]:
# =========================================================
# 8. 누적 비중 기준 목적지 선택 함수
# =========================================================

def select_destinations_by_threshold(
    od_df: pd.DataFrame,
    threshold: float,
) -> pd.DataFrame:
    """
    거주동별로 누적 출근 비중이 threshold 이상이 되는
    최초 목적지까지 선택한다.

    예:
    threshold=0.8
    누적 비중이 0.77, 0.82이면 0.82 행까지 포함
    """

    result_list = []

    for residence_code, group in od_df.groupby(
        "거주동 코드",
        sort=False,
        observed=True,
    ):
        group = (
            group
            .sort_values(
                [
                    "출근_이동량",
                    "근무동 코드",
                ],
                ascending=[
                    False,
                    True,
                ],
            )
            .copy()
        )

        # 누적 비중이 기준 이상이 되는 첫 번째 행 위치
        reached = group["누적_출근비중"] >= threshold

        if reached.any():
            cutoff_position = np.flatnonzero(
                reached.to_numpy()
            )[0]

            selected_group = group.iloc[
                :cutoff_position + 1
            ].copy()

        else:
            # 부동소수점 오차 등으로 기준에 도달하지 못하면 전체 선택
            selected_group = group.copy()

        result_list.append(selected_group)

    selected = pd.concat(
        result_list,
        ignore_index=True,
    )

    selected["선택_기준"] = threshold

    return selected

In [10]:
# =========================================================
# 9. 기준별 주요 목적지 추출
# =========================================================

selected_70 = select_destinations_by_threshold(
    od,
    threshold=0.70,
)

selected_80 = select_destinations_by_threshold(
    od,
    threshold=0.80,
)

selected_90 = select_destinations_by_threshold(
    od,
    threshold=0.90,
)


print("70% 기준 선택 OD 수:", f"{len(selected_70):,}")
print("80% 기준 선택 OD 수:", f"{len(selected_80):,}")
print("90% 기준 선택 OD 수:", f"{len(selected_90):,}")

70% 기준 선택 OD 수: 20,112
80% 기준 선택 OD 수: 30,839
90% 기준 선택 OD 수: 51,328


In [11]:
# =========================================================
# 10. 선택 목적지 안에서 최종 가중치 재계산
# =========================================================

def add_normalized_weight(
    selected_df: pd.DataFrame,
) -> pd.DataFrame:
    result = selected_df.copy()

    result["선택목적지_출근량합"] = (
        result
        .groupby(
            "거주동 코드",
            observed=True,
        )["출근_이동량"]
        .transform("sum")
    )

    result["최종_가중치"] = (
        result["출근_이동량"]
        / result["선택목적지_출근량합"]
    )

    return result


selected_70 = add_normalized_weight(selected_70)
selected_80 = add_normalized_weight(selected_80)
selected_90 = add_normalized_weight(selected_90)

In [12]:
# =========================================================
# 11. 최종 가중치 검증
# =========================================================

def check_normalized_weight(
    selected_df: pd.DataFrame,
    label: str,
) -> pd.DataFrame:

    check = (
        selected_df
        .groupby(
            [
                "거주동 코드",
                "거주동 이름",
            ],
            as_index=False,
            observed=True,
        )
        .agg(
            최종_가중치합=("최종_가중치", "sum"),
            선택_OD수=("근무동 코드", "nunique"),
            선택_출근량=("출근_이동량", "sum"),
            전체_출근량=("거주동_전체_출근량", "first"),
            선택_누적비중=("목적지_출근비중", "sum"),
        )
    )

    check["가중치합_오차"] = (
        check["최종_가중치합"] - 1
    ).abs()

    print(f"\n[{label} 기준]")
    print(
        "최종 가중치 합계 이상 거주동 수:",
        (
            check["가중치합_오차"] > 1e-8
        ).sum()
    )

    return check


check_70 = check_normalized_weight(
    selected_70,
    "70%",
)

check_80 = check_normalized_weight(
    selected_80,
    "80%",
)

check_90 = check_normalized_weight(
    selected_90,
    "90%",
)


[70% 기준]
최종 가중치 합계 이상 거주동 수: 0

[80% 기준]
최종 가중치 합계 이상 거주동 수: 0

[90% 기준]
최종 가중치 합계 이상 거주동 수: 0


In [13]:
# =========================================================
# 12. 기준별 API 호출량 비교
# =========================================================

def summarize_threshold(
    selected_df: pd.DataFrame,
    threshold_label: str,
) -> dict:

    residence_summary = (
        selected_df
        .groupby(
            "거주동 코드",
            observed=True,
        )
        .agg(
            선택_OD수=("근무동 코드", "nunique"),
            포함_출근비중=("목적지_출근비중", "sum"),
        )
    )

    return {
        "누적비중_기준": threshold_label,
        "전체_선택_OD수": len(selected_df),
        "거주동수": selected_df["거주동 코드"].nunique(),
        "거주동당_평균_OD수": residence_summary[
            "선택_OD수"
        ].mean(),
        "거주동당_중앙값_OD수": residence_summary[
            "선택_OD수"
        ].median(),
        "거주동당_최소_OD수": residence_summary[
            "선택_OD수"
        ].min(),
        "거주동당_최대_OD수": residence_summary[
            "선택_OD수"
        ].max(),
        "평균_포함출근비중": residence_summary[
            "포함_출근비중"
        ].mean(),
    }


threshold_summary = pd.DataFrame(
    [
        summarize_threshold(
            selected_70,
            "70%",
        ),
        summarize_threshold(
            selected_80,
            "80%",
        ),
        summarize_threshold(
            selected_90,
            "90%",
        ),
    ]
)


threshold_summary[
    "거주동당_평균_OD수"
] = threshold_summary[
    "거주동당_평균_OD수"
].round(2)

threshold_summary[
    "평균_포함출근비중"
] = threshold_summary[
    "평균_포함출근비중"
].round(4)


display(threshold_summary)

,누적비중_기준,전체_선택_OD수,거주동수,거주동당_평균_OD수,거주동당_중앙값_OD수,거주동당_최소_OD수,거주동당_최대_OD수,평균_포함출근비중
0,70%,20112,428,46.99,46.0,7,80,0.7028
1,80%,30839,428,72.05,71.0,18,115,0.8015
2,90%,51328,428,119.93,117.0,47,173,0.9007


In [14]:
# =========================================================
# 13. 기준별 기존 대표 이동시간 계산
# =========================================================

def calculate_current_representative_values(
    selected_df: pd.DataFrame,
    threshold_label: str,
) -> pd.DataFrame:

    temp = selected_df.copy()

    temp["가중_이동시간"] = (
        temp["평균_이동시간_분"]
        * temp["최종_가중치"]
    )

    temp["가중_이동거리"] = (
        temp["평균_이동거리_m"]
        * temp["최종_가중치"]
    )

    result = (
        temp
        .groupby(
            [
                "거주동 코드",
                "거주동 이름",
            ],
            as_index=False,
            observed=True,
        )
        .agg(
            대표_이동시간_분=(
                "가중_이동시간",
                "sum",
            ),
            대표_이동거리_m=(
                "가중_이동거리",
                "sum",
            ),
            선택_OD수=(
                "근무동 코드",
                "nunique",
            ),
            포함_출근비중=(
                "목적지_출근비중",
                "sum",
            ),
        )
    )

    result["기준"] = threshold_label

    return result


representative_70 = (
    calculate_current_representative_values(
        selected_70,
        "70%",
    )
)

representative_80 = (
    calculate_current_representative_values(
        selected_80,
        "80%",
    )
)

representative_90 = (
    calculate_current_representative_values(
        selected_90,
        "90%",
    )
)

In [15]:
# =========================================================
# 14. 기준별 대표값 차이 비교
# =========================================================

comparison = (
    representative_70[
        [
            "거주동 코드",
            "거주동 이름",
            "대표_이동시간_분",
            "대표_이동거리_m",
            "선택_OD수",
        ]
    ]
    .rename(
        columns={
            "대표_이동시간_분": "대표시간_70",
            "대표_이동거리_m": "대표거리_70",
            "선택_OD수": "OD수_70",
        }
    )
    .merge(
        representative_80[
            [
                "거주동 코드",
                "대표_이동시간_분",
                "대표_이동거리_m",
                "선택_OD수",
            ]
        ].rename(
            columns={
                "대표_이동시간_분": "대표시간_80",
                "대표_이동거리_m": "대표거리_80",
                "선택_OD수": "OD수_80",
            }
        ),
        on="거주동 코드",
        how="outer",
    )
    .merge(
        representative_90[
            [
                "거주동 코드",
                "대표_이동시간_분",
                "대표_이동거리_m",
                "선택_OD수",
            ]
        ].rename(
            columns={
                "대표_이동시간_분": "대표시간_90",
                "대표_이동거리_m": "대표거리_90",
                "선택_OD수": "OD수_90",
            }
        ),
        on="거주동 코드",
        how="outer",
    )
)


comparison["시간차_80_70"] = (
    comparison["대표시간_80"]
    - comparison["대표시간_70"]
).abs()

comparison["시간차_90_80"] = (
    comparison["대표시간_90"]
    - comparison["대표시간_80"]
).abs()

comparison["추가_OD수_80_70"] = (
    comparison["OD수_80"]
    - comparison["OD수_70"]
)

comparison["추가_OD수_90_80"] = (
    comparison["OD수_90"]
    - comparison["OD수_80"]
)


print("70% → 80% 평균 대표시간 차이")
print(
    comparison["시간차_80_70"].mean().round(2),
    "분"
)

print("\n80% → 90% 평균 대표시간 차이")
print(
    comparison["시간차_90_80"].mean().round(2),
    "분"
)

print("\n70% → 80% 평균 추가 OD 수")
print(
    comparison["추가_OD수_80_70"].mean().round(2)
)

print("\n80% → 90% 평균 추가 OD 수")
print(
    comparison["추가_OD수_90_80"].mean().round(2)
)

display(
    comparison
    .sort_values(
        "시간차_90_80",
        ascending=False,
    )
    .head(20)
)

70% → 80% 평균 대표시간 차이
1.48 분

80% → 90% 평균 대표시간 차이
1.45 분

70% → 80% 평균 추가 OD 수
25.06

80% → 90% 평균 추가 OD 수
47.87


,거주동 코드,거주동 이름,대표시간_70,대표거리_70,OD수_70,대표시간_80,대표거리_80,OD수_80,대표시간_90,대표거리_90,OD수_90,시간차_80_70,시간차_90_80,추가_OD수_80_70,추가_OD수_90_80
387,11710550,마천2동,33.534150,3969.770044,39,36.462057,4643.066339,58,39.537194,5506.970186,90,2.927908,3.075137,19,32
8,11110615,종로1.2.3.4가동,17.887047,848.520538,11,19.677306,1254.031202,27,22.658226,1886.332267,78,1.790260,2.980920,16,51
17,11140520,소공동,14.901423,849.916366,7,16.484821,1151.961608,18,19.450420,1761.510586,57,1.583398,2.965599,11,39
260,11500640,방화2동,33.380657,5491.441289,30,36.433909,6462.463068,51,39.362170,7520.174842,90,3.053252,2.928260,21,39
7,11110600,가회동,25.875851,2703.113082,25,28.190754,3259.181865,41,31.008418,3868.903924,72,2.314903,2.817664,16,31
231,11470580,신월3동,37.787889,4762.245182,43,40.879991,5643.427081,64,43.668977,6406.101808,100,3.092102,2.788986,21,36
258,11500620,공항동,31.288447,5173.380808,30,33.529189,5852.750246,49,36.278599,6692.192926,91,2.240742,2.749409,19,42
19,11140550,명동,14.008902,871.065346,10,16.639801,1382.154909,27,19.365159,2037.244601,72,2.630899,2.725358,17,45
256,11500611,발산1동,31.636546,4650.275346,35,34.308542,5381.002790,57,36.979316,6155.436924,101,2.671995,2.670775,22,44
377,11680690,개포4동,29.254853,3338.807579,28,31.614014,3918.136855,46,34.270425,4582.612194,92,2.359161,2.656410,18,46


In [16]:
# =========================================================
# 15. 저장할 컬럼 정리
# =========================================================

FINAL_COLS = [
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "목적지_순위",
    "출근_이동량",
    "거주동_전체_출근량",
    "목적지_출근비중",
    "누적_출근비중",
    "선택목적지_출근량합",
    "최종_가중치",
    "평균_이동시간_분",
    "평균_이동거리_m",
]

# 기존 파일에 km 컬럼이 있으면 함께 저장
if "평균_이동거리_km" in od.columns:
    FINAL_COLS.append("평균_이동거리_km")


selected_70 = selected_70[FINAL_COLS].copy()
selected_80 = selected_80[FINAL_COLS].copy()
selected_90 = selected_90[FINAL_COLS].copy()

In [17]:
# =========================================================
# 16. 소수점 정리
# =========================================================

def round_selected_result(
    selected_df: pd.DataFrame,
) -> pd.DataFrame:

    result = selected_df.copy()

    result["출근_이동량"] = (
        result["출근_이동량"]
        .round(2)
    )

    result["거주동_전체_출근량"] = (
        result["거주동_전체_출근량"]
        .round(2)
    )

    result["목적지_출근비중"] = (
        result["목적지_출근비중"]
        .round(8)
    )

    result["누적_출근비중"] = (
        result["누적_출근비중"]
        .round(8)
    )

    result["선택목적지_출근량합"] = (
        result["선택목적지_출근량합"]
        .round(2)
    )

    result["최종_가중치"] = (
        result["최종_가중치"]
        .round(8)
    )

    return result


selected_70 = round_selected_result(selected_70)
selected_80 = round_selected_result(selected_80)
selected_90 = round_selected_result(selected_90)

In [18]:
# =========================================================
# 17. 결과 저장
# =========================================================

OUTPUT_70 = (
    PROCESSED_DIR
    / "all_age_commute_od_selected_70.csv"
)

OUTPUT_80 = (
    PROCESSED_DIR
    / "all_age_commute_od_selected_80.csv"
)

OUTPUT_90 = (
    PROCESSED_DIR
    / "all_age_commute_od_selected_90.csv"
)

SUMMARY_FILE = (
    PROCESSED_DIR
    / "commute_destination_threshold_summary.csv"
)

COMPARISON_FILE = (
    PROCESSED_DIR
    / "commute_destination_threshold_comparison.csv"
)


selected_70.to_csv(
    OUTPUT_70,
    index=False,
    encoding="utf-8-sig",
)

selected_80.to_csv(
    OUTPUT_80,
    index=False,
    encoding="utf-8-sig",
)

selected_90.to_csv(
    OUTPUT_90,
    index=False,
    encoding="utf-8-sig",
)

threshold_summary.to_csv(
    SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig",
)

comparison.to_csv(
    COMPARISON_FILE,
    index=False,
    encoding="utf-8-sig",
)


print("70% 결과 저장:", OUTPUT_70)
print("80% 결과 저장:", OUTPUT_80)
print("90% 결과 저장:", OUTPUT_90)
print("기준별 요약 저장:", SUMMARY_FILE)
print("기준별 비교 저장:", COMPARISON_FILE)

70% 결과 저장: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_od_selected_70.csv
80% 결과 저장: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_od_selected_80.csv
90% 결과 저장: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_od_selected_90.csv
기준별 요약 저장: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_destination_threshold_summary.csv
기준별 비교 저장: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_destination_threshold_comparison.csv


In [19]:
# =========================================================
# 18. 80% 결과 최종 확인
# =========================================================

print("=" * 60)
print("80% 기준 주요 출근 목적지 결과")
print("=" * 60)

print("선택 OD 수:", f"{len(selected_80):,}")
print(
    "거주동 수:",
    f"{selected_80['거주동 코드'].nunique():,}"
)

print(
    "거주동당 평균 선택 목적지 수:",
    round(
        selected_80
        .groupby("거주동 코드")
        .size()
        .mean(),
        2,
    )
)

weight_check_80 = (
    selected_80
    .groupby("거주동 코드")["최종_가중치"]
    .sum()
)

print(
    "최종 가중치 합 최소:",
    weight_check_80.min()
)

print(
    "최종 가중치 합 최대:",
    weight_check_80.max()
)

display(selected_80.head(30))

80% 기준 주요 출근 목적지 결과
선택 OD 수: 30,839
거주동 수: 428
거주동당 평균 선택 목적지 수: 72.05
최종 가중치 합 최소: 0.99999993
최종 가중치 합 최대: 1.00000007


,거주동 코드,거주동 이름,근무동 코드,근무동 이름,목적지_순위,출근_이동량,거주동_전체_출근량,목적지_출근비중,누적_출근비중,선택목적지_출근량합,최종_가중치,평균_이동시간_분,평균_이동거리_m,평균_이동거리_km
0,11110515,청운효자동,11110530,사직동,1,55710.25,564412.04,0.098705,0.098705,451971.51,0.123261,17.68,928.20,0.928
1,11110515,청운효자동,11110615,종로1.2.3.4가동,2,47373.52,564412.04,0.083934,0.182639,451971.51,0.104815,25.72,1759.83,1.760
2,11110515,청운효자동,11110515,청운효자동,3,43228.92,564412.04,0.076591,0.259230,451971.51,0.095645,15.46,502.69,0.503
3,11110515,청운효자동,11140550,명동,4,21721.17,564412.04,0.038485,0.297715,451971.51,0.048059,30.46,2213.62,2.214
4,11110515,청운효자동,11140520,소공동,5,17247.38,564412.04,0.030558,0.328273,451971.51,0.038160,30.19,2127.88,2.128
5,11110515,청운효자동,11560540,여의동,6,17222.83,564412.04,0.030515,0.358788,451971.51,0.038106,40.96,7382.10,7.382
6,11110515,청운효자동,11680640,역삼1동,7,13614.66,564412.04,0.024122,0.382910,451971.51,0.030123,57.33,10844.29,10.844
7,11110515,청운효자동,11140540,회현동,8,13283.24,564412.04,0.023535,0.406444,451971.51,0.029390,32.23,2790.27,2.790
8,11110515,청운효자동,11110600,가회동,9,11631.69,564412.04,0.020609,0.427053,451971.51,0.025735,23.67,1752.49,1.752
9,11110515,청운효자동,11110640,이화동,10,11112.00,564412.04,0.019688,0.446740,451971.51,0.024586,37.80,3151.91,3.152
